# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata properties directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. The `mlcroissant` library provides access to record sets defined in the Croissant schema. All entities are referenced by their `@id`s.

Let's enumerate record sets and list their fields and columns.

In [ ]:
# Get all record sets by @id
record_sets = dataset.record_sets()
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Record Set @id: {rs_id}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - {f['@id']} : {f.get('name', f.get('@id'))}")
            columns = f.get('column', [])
            if isinstance(columns, dict):
                columns = [columns]
            if columns:
                print("      Columns:")
                for c in columns:
                    print(f"        * {c['@id']} : {c.get('name', c.get('@id'))}")
    else:
        print("  No fields found.")
    print("")

## 3. Data Extraction

Now, load records from a specific record set into a DataFrame using its `@id`. We use the list collected previously. Each DataFrame will use record set and field `@id`s as column names.

First, let's prepare the list of record set `@id`s for extraction.

In [ ]:
# List all record set @id's and load each as a DataFrame
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Extract records; each record should be keyed by field @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in DataFrame [{first_rs_id}]:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical processing steps, such as filtering, normalization, grouping, and data cleaning. All fields and columns are referenced by their `@id`.

We select some numeric and group fields based on the record set schema. Adapt this section with record set, field, and column `@id`s accordingly.

In [ ]:
# Replace with a relevant numeric field @id from the data overview
# For illustration, use a likely field name such as 'age' or similar
# Assume the first record set has fields including '@id': 'http://senscience.ai/age' and 'http://senscience.ai/sex'

rs_id = record_set_ids[0] if record_set_ids else None
if rs_id and not dataframes[rs_id].empty:
    df = dataframes[rs_id]
    # Try to find a numeric field
    possible_numeric = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'metastasis' in col.lower()]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Try grouping by a categorical field, e.g., sex or anatomical location
        possible_group = [col for col in df.columns if 'sex' in col.lower() or 'location' in col.lower()]
        if possible_group:
            group_field_id = possible_group[0]
            print(f"Grouping by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No suitable record set or empty DataFrame for EDA.")

## 5. Visualization

Visualize distributions or relationships between fields using their `@id`. We use Matplotlib for visualization.

In [ ]:
# Visualize numeric field distribution and relationship to grouping field
if rs_id and not dataframes[rs_id].empty and 'numeric_field_id' in locals():
    df = dataframes[rs_id]
    plt.figure(figsize=(7,5))
    df[numeric_field_id].hist(bins=15)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()
    
    if 'group_field_id' in locals():
        plt.figure(figsize=(7,5))
        df.groupby(group_field_id)[numeric_field_id].mean().plot(kind='bar')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

We successfully loaded the FAIR^2 dataset package via its Croissant schema, explored its structure using `mlcroissant`, inspected fields and columns using their `@id`s, and performed basic exploratory data analysis and visualization.

This approach ensures FAIR data referencing and processing. For any downstream analytics, refer to each field and record set by its `@id` as defined in the Croissant schema.

**Next steps:** Use this dataset for model development, advanced stratification studies, or integrate with clinical pipelines using reproducible FAIR principles.